# Count a shape as it grows
### Lattice points, finite differences, and an entrance to Ehrhart theory

An integer-coordinate triangle grows one integer step at a time. Its counts begin
$1,3,6,10,\ldots$. What changes when we count the changes themselves? And what
breaks if the triangle's vertices have half-integer coordinates?

This notebook keeps each measured parameter case as a symbolic construction. It
then derives difference arrangements and uses their values to move independent
points. A final example reveals a periodic pattern that one polynomial cannot describe.

[Lesson notes](../docs/lessons/08_ehrhart_counts.md) · [Setup](README.md)

In [ ]:
from pathlib import Path
from functools import reduce
import json
import sys
import numpy as np
import plotly.io as pio
from IPython.display import Video, display
from kaleion import Collection, F, Motion, Workspace, param
from kaleion.viewers.plotly import animation_figure
from kaleion.viewers.video import write_mp4

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "pyproject.toml").exists() and (p / "src/kaleion").is_dir())
sys.path.insert(0, str(ROOT / "notebooks"))
from lesson_views import COLORS, profiles, replay, save_figures
pio.renderers.default = "plotly_mimetype+notebook"
OUTPUT = ROOT / "build/notebooks/ehrhart-counts"
OUTPUT.mkdir(parents=True, exist_ok=True)

MAX_N = 8
assert isinstance(MAX_N, int) and not isinstance(MAX_N, bool) and 4 <= MAX_N <= 12
n = param("n")

## 1 · Declare exact integer dilations

For $n\ge0$, count the integer points in

\[
T_n=\{(i,j): i,j\ge0,\ i+j\le n\}.
\]

The finite grid below contains every point needed for $0\le n\le\texttt{MAX_N}$.
Dim points in the viewer are outside the selected triangle, not missing data.
The scrubber selects exact integer cases; it does not invent intermediate lattice counts.

In [ ]:
domain = Collection.grid(MAX_N + 1, MAX_N + 1, values=1).arrange(F.i, F.j)
triangle = domain.where(F.i + F.j <= n)
interior = domain.where((F.i > 0) & (F.j > 0) & (F.i + F.j < n))
rational_triangle = domain.where(2 * (F.i + F.j) <= n)

def measured_family(incidence):
    total = incidence.count()
    cases = {k: total.with_params(n=k).annotate(scale=k) for k in range(MAX_N + 1)}
    # Keep definitions as inputs: no evaluated numbers are reinserted as literals.
    family = reduce(lambda left, right: left.concat(right), cases.values())
    return family.annotate(key=F.scale).arrange(F.scale, F.value), cases

counts, count_cases = measured_family(triangle)
interior_counts, interior_cases = measured_family(interior)
rational_counts, rational_cases = measured_family(rational_triangle)
triangle_cases = {k: triangle.with_params(n=k) for k in range(MAX_N + 1)}

## 2 · Turn a sequence of counts into a sequence of differences

\[
\Delta L(n)=L(n+1)-L(n),\qquad
\Delta^2L(n)=\Delta L(n+1)-\Delta L(n).
\]

Each lookup aligns by the explicit scale key. We restrict the target to keys that
have a next case. These are exact differences between integer measurements;
they do not differentiate animation frames.

In [ ]:
def difference(series, last, *, step=1):
    available = series.where(F.scale <= last - step).select()
    return available.with_values(series.bind(on=F.scale + step, key=F.scale) - F.value).arrange(F.scale, F.value)

first = difference(counts, MAX_N)
second = difference(first, MAX_N - 1)
rational_first = difference(rational_counts, MAX_N)
rational_second = difference(rational_first, MAX_N - 1)
stride_two = difference(rational_counts, MAX_N, step=2)
stride_two_second = difference(stride_two, MAX_N - 2, step=2)
boundary_counts = counts.with_values(F.value - interior_counts.bind(on=F.scale, key=F.scale))

ROOTS = {"counts": counts, "first": first, "second": second,
         "interior": interior_counts, "boundary": boundary_counts,
         "rational": rational_counts, "rational_second": rational_second,
         "stride_two_second": stride_two_second,
         **{f"count_{k}": obj for k, obj in count_cases.items()},
         **{f"triangle_{k}": obj for k, obj in triangle_cases.items()}}
workspace = Workspace(ROOTS)
assert not workspace.state.errors, dict(workspace.state.errors)
state = workspace.state
r = state.results
assert r["counts"].values.tolist() == [(k + 1) * (k + 2) // 2 for k in range(MAX_N + 1)]
assert r["first"].values.tolist() == list(range(2, MAX_N + 2))
assert r["second"].values.tolist() == [1] * (MAX_N - 1)
case_snapshots = [r[f"triangle_{k}"] for k in range(MAX_N + 1)]
case_labels = [f"n={k} · L(n)={snapshot.cardinality}" for k, snapshot in enumerate(case_snapshots)]
dilation_plot = animation_figure(case_snapshots, labels=case_labels,
    title="Integer dilations · count the selected lattice points", duration=650, show_values=False)
dilation_plot.show()
count_plot = profiles([r["counts"], r["first"], r["second"]],
    ["L(n)", "First differences", "Second differences"], keys=["scale"] * 3,
    title="The counts grow quadratically · their second differences are constant")
count_plot.show()

## 3 · Let those measurements move another arrangement

Construct new probe points on the common scale keys $0,\ldots,\texttt{MAX_N}-2$.
Move them to heights $L(n)$, then $\Delta L(n)$, then $\Delta^2L(n)$.
The final points form a horizontal line at height one. Each driver is a derived
arrangement whose construction still refers to the original incidence cases.

The axes remain fixed during playback, with independent screen scales. Their visual
aspect ratio is not a claim about geometric distances. Undo restores the same
captured states and paths.

In [ ]:
probes = (Collection.sequence(MAX_N - 1, start=0).annotate(scale=F.value)
          .with_values(0).arrange(F.scale, 0))
stages = [probes.with_values(driver.bind(on=F.scale, key=F.scale)).arrange(F.scale, F.value)
          for driver in (counts, first, second)]
motion_workspace = Workspace({"probes": probes})
outward = [motion_workspace.set("probes", stage, motion=Motion()) for stage in stages]
assert not motion_workspace.state.errors
assert set(motion_workspace.state.results["probes"].values) == {1}
assert outward[0].start.results["probes"].ids == motion_workspace.state.results["probes"].ids
motion_workspace.capture("Case measurements and their finite differences drive these independent probe points.")
inward = [motion_workspace.undo() for _ in outward]
samples, captions = [], []
names = ["move to measured counts", "move to first differences", "move to second differences"]
for name, transition in zip(names + ["undo · " + name for name in reversed(names)], outward + inward):
    for progress in np.linspace(0, 1, 17):
        samples.append(transition.frame("probes", float(progress)))
        captions.append(f"{name} · {progress:.0%}")
np.testing.assert_array_equal(samples[0].positions, samples[-1].positions)
for forward, backward in zip(outward, reversed(inward)):
    np.testing.assert_array_equal(forward.frame("probes", .25).positions, backward.frame("probes", .75).positions)
probe_source = outward[0].start.results["probes"]
colors = {oid: COLORS[int(scale) % len(COLORS)] for oid, scale in zip(probe_source.ids, probe_source.fields["scale"])}
motion_plot = replay(samples, captions, title="From counts to differences · the same probes follow three measured fields", colors_by_id=colors)
motion_plot.update_yaxes(scaleanchor=None, title_text="measured value")
motion_plot.update_xaxes(title_text="scale n")
motion_plot.show()

## 4 · Explain the polynomial, then look inside the boundary

At a fixed $i$, the allowed $j$ values are $0,\ldots,n-i$. Thus

\[
L(n)=\sum_{i=0}^n(n-i+1)=\frac{(n+1)(n+2)}2.
\]

This row-counting argument proves the formula for every integer $n\ge0$; a finite
table of differences alone would not prove it. Its quadratic coefficient $1/2$
is the Euclidean area of the unit triangle. More generally, integer dilations of a
lattice polytope have a polynomial lattice-point count, the **Ehrhart polynomial**.

For $n\ge1$, the strict interior requires $i,j>0$ and $i+j<n$, giving
$I(n)=(n-1)(n-2)/2$. For $n=1,2$ the interior is empty. Consequently

\[
L(-n)=\frac{(1-n)(2-n)}2=I(n),\qquad n\ge1.
\]

This is the triangle's instance of Ehrhart reciprocity. We evaluate a polynomial
at a negative integer; we do not claim that a reflected triangle loses its boundary.
At $n=0$, the closed triangle is one point and our strict-inequality lens is empty;
the displayed reciprocity comparison deliberately excludes this collapsed case.

For the general theorem and its context, see
[Coefficients and Roots of Ehrhart Polynomials](https://math.mit.edu/~rstan/papers/ehrhart.pdf).
The elementary arguments here establish the two formulas for this particular triangle.

In [ ]:
negative_polynomial = counts.where(F.scale > 0).select().with_values((1 - F.scale) * (2 - F.scale) // 2)
reciprocity_residual = negative_polynomial.with_values(
    F.value - interior_counts.bind(on=F.scale, key=F.scale))
workspace.set("negative_polynomial", negative_polynomial)
workspace.set("reciprocity_residual", reciprocity_residual)
assert not workspace.state.errors
assert set(workspace.state.results["reciprocity_residual"].values) == {0}
assert r["interior"].values.tolist() == [0] + [(k - 1) * (k - 2) // 2 for k in range(1, MAX_N + 1)]
assert r["boundary"].values.tolist() == [1] + [3 * k for k in range(1, MAX_N + 1)]
boundary_plot = profiles([r["counts"], r["interior"], r["boundary"]],
    ["Closed triangle", "Strict interior", "Boundary count"], keys=["scale"] * 3,
    title="Boundary conventions change the counting polynomial")
boundary_plot.show()
print("Reciprocity residuals for n=1,...,MAX_N:", workspace.state.results["reciprocity_residual"].values.tolist())

## 5 · Half-integer vertices reveal periodicity

Now start with the triangle whose vertices are $(0,0),(1/2,0),(0,1/2)$. Its n-th
dilation selects integer points satisfying $2(i+j)\le n$. Let $q=\lfloor n/2\rfloor$.
The same row argument gives

\[
L_Q(n)=\frac{(q+1)(q+2)}2
=\begin{cases}
\dfrac{(n+2)(n+4)}8,&n\text{ even},\\[3pt]
\dfrac{(n+1)(n+3)}8,&n\text{ odd}.
\end{cases}
\]

The counts begin $1,1,3,3,6,6,10,10,\ldots$. Consecutive second differences are not
constant. Compare cases two steps apart instead: each residue class has constant
second differences. A polynomial formula selected by a residue class is a
**quasipolynomial**. All boundary predicates here still use exact integer arithmetic.

These two different polynomial formulas cannot be replaced by a single polynomial
valid at every nonnegative integer: agreement on the infinitely many even inputs
would force that polynomial to equal the even formula everywhere, which fails on odd inputs.

In [ ]:
assert r["rational"].values.tolist() == [(k // 2 + 1) * (k // 2 + 2) // 2 for k in range(MAX_N + 1)]
assert len(set(r["rational_second"].values)) > 1
assert set(r["stride_two_second"].values) == {1}
rational_plot = profiles([r["rational"], r["rational_second"], r["stride_two_second"]],
    ["Half-triangle counts", "Δ² · stride 1", "Δ² · stride 2"],
    keys=["scale"] * 3, title="A periodic counting law · separate even and odd scales")
rational_plot.show()
print("Rational triangle counts:", r["rational"].values.tolist())
print("Second differences with stride 2:", r["stride_two_second"].values.tolist())

## 6 · Follow a count back to its parameter case

Concatenation preserves definitions and parent lineage, but the assembled sequence
does not expose every scalar reduction's contributor lookup as its own metadata.
We therefore retain the individual count cases as workspace roots and inspect them
by their scale key. A scalar reduction uses the empty grouping key `()`.

The explanation below links one profile occurrence, its measured case, and the original
lattice points. A future parameter-family convenience layer should make this
inspection easier without replacing definitions with a list of evaluated values.

In [ ]:
INSPECT_N = min(4, MAX_N)
case_count = state.results[f"count_{INSPECT_N}"]
case_incidence = state.results[f"triangle_{INSPECT_N}"]
source = case_incidence.source
by_id = {oid: [int(source.fields[axis][i]) for axis in ("i", "j")]
         for i, oid in enumerate(source.ids)}
contributors = case_count.contributor_ids(())
assert set(contributors) == {oid for oid, selected in zip(source.ids, case_incidence.mask) if selected}
profile_index = list(map(int, r["counts"].fields["scale"])).index(INSPECT_N)
explanation = {"scale": INSPECT_N, "count": int(case_count.values[0]),
               "profile_occurrence": r["counts"].ids[profile_index], "count_occurrence": case_count.ids[0],
               "contributors": [{"occurrence": oid, "point": by_id[oid]} for oid in contributors]}
print("At scale", INSPECT_N, "the count is", explanation["count"])
print("Original points:", [item["point"] for item in explanation["contributors"]])

# Hold exact cases; repeated snapshots control pacing, not mathematical sampling.
video_samples = [snapshot for snapshot in case_snapshots for _ in range(8)]
video_labels = [label for label in case_labels for _ in range(8)]
movie = write_mp4(video_samples, OUTPUT / "integer-dilations.mp4", labels=video_labels,
                   title="Exact lattice-point counts at integer dilations", fps=12)
display(Video(str(movie), embed=True))

workspace.capture("A family of measured integer dilations yields exact difference arrangements and a reciprocity comparison.")
save_figures(OUTPUT, {"integer-dilations": dilation_plot, "count-differences": count_plot,
    "measured-difference-motion": motion_plot, "interior-boundary": boundary_plot, "rational-periodicity": rational_plot})
for name, investigation in (("counts", workspace), ("motion", motion_workspace)):
    payload = investigation.to_json()
    (OUTPUT / f"{name}-workspace.json").write_text(payload)
    reopened = Workspace.from_json(payload)
    assert not reopened.state.errors
restored = Workspace.from_json(workspace.to_json())
assert restored.state.results[f"count_{INSPECT_N}"].contributor_ids(()) == contributors
(OUTPUT / "case-explanation.json").write_text(json.dumps(explanation, indent=2))
(OUTPUT / "checks.json").write_text(json.dumps({"maximum_scale": MAX_N,
    "counts": list(map(int, r["counts"].values)), "first": list(map(int, r["first"].values)),
    "second": list(map(int, r["second"].values)), "rational": list(map(int, r["rational"].values)),
    "stride_two_second": list(map(int, r["stride_two_second"].values)),
    "motion_frames": len(samples), "video_frames": len(video_samples)}, indent=2))
print("Saved five offline figures, the exact-case MP4, workspaces, and a parameter-case explanation to", OUTPUT)

The recurring construction is **parameter cases → measured family → key-aligned
differences → a new driven arrangement**. The proof supplies the general statement;
the notebook lets us observe, challenge, and inspect particular cases.

The [review notes](../docs/lessons/REVIEW_NOTES.md) collect what lessons 01–08 reveal
about Kaleion's current vocabulary, ergonomics, and possible next steps.